# Swimming Pool Detection from Aerial Images
### MBD Individual Assignment 2026

This notebook covers the full assignment pipeline:

| Step | Task |
|------|------|
| 1 | Auto-annotation with GroundingDINO + manual review in Roboflow |
| 2 | YOLO26 training and scaling comparison (n / s / m / l / x) |
| 3 | RF-DETR training and benchmarking against YOLO26 |
| 4 | Oriented Bounding Box (OBB) detection with YOLO26-OBB and YOLO11-OBB |

The six required discussion questions and the failure case analysis are answered at the end.

---

### How to run this notebook locally

1. Clone the repository — it contains `images/` (all images) and `labels/` (YOLO-format `.txt` files).
2. Make sure you have a GPU available locally (`nvidia-smi` to check).
3. Install dependencies once (the `pip install` cells below).
4. Run the setup cell — it automatically splits the flat `images/` and `labels/` folders 80/20 into `dataset/` and generates `data.yaml`.
5. Train one model at a time. There is a memory cleanup cell after each training block — run it before the next model.

All outputs (weights, CSVs, figures) are written to `runs/` and `results/` inside the repo.

## 0. Setup (run once)

GPU check, library install, imports, and project paths.

The RF-DETR library has heavier dependencies and is installed separately in Step 3.

In [ ]:
import os
# Helps avoid GPU memory fragmentation when training large models.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Check GPU availability
import subprocess
try:
    print(subprocess.check_output('nvidia-smi', shell=True).decode())
except Exception:
    print('No nvidia-smi found. Running on CPU — training will be slow.')

In [ ]:
%pip install -q 'ultralytics>=8.4.0' supervision
import ultralytics
ultralytics.checks()

In [ ]:
import os, gc, glob, random, json, shutil
from pathlib import Path
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Image as IPyImage, display
from ultralytics import YOLO

# ── Project paths ────────────────────────────────────────────────────────────
# Repo root — images/ and labels/ live directly here.
PROJECT_ROOT = os.path.abspath('.')

DATASET_PATH = f'{PROJECT_ROOT}/dataset'
DATA_YAML    = f'{PROJECT_ROOT}/data.yaml'
RUNS_DIR     = f'{PROJECT_ROOT}/runs'
RESULTS_DIR  = f'{PROJECT_ROOT}/results'

TRAIN_IMAGES = f'{DATASET_PATH}/images/train'
VAL_IMAGES   = f'{DATASET_PATH}/images/val'
TRAIN_LABELS = f'{DATASET_PATH}/labels/train'
VAL_LABELS   = f'{DATASET_PATH}/labels/val'

os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Build 80/20 train/val split from flat images/ and labels/ if not already done.
_src_imgs = Path(PROJECT_ROOT) / 'images'
_src_lbls = Path(PROJECT_ROOT) / 'labels'
if _src_imgs.exists() and not (Path(TRAIN_IMAGES).exists() and any(Path(TRAIN_IMAGES).glob('*'))):
    _all = sorted(_src_imgs.glob('*.jpg')) + sorted(_src_imgs.glob('*.jpeg')) + sorted(_src_imgs.glob('*.png'))
    import random as _rnd; _rnd.seed(42); _rnd.shuffle(_all)
    _n = int(len(_all) * 0.8)
    for _split, _imgs in [('train', _all[:_n]), ('val', _all[_n:])]:
        (Path(DATASET_PATH) / 'images' / _split).mkdir(parents=True, exist_ok=True)
        (Path(DATASET_PATH) / 'labels' / _split).mkdir(parents=True, exist_ok=True)
        for _img in _imgs:
            shutil.copy2(_img, Path(DATASET_PATH) / 'images' / _split / _img.name)
            _lbl = _src_lbls / (_img.stem + '.txt')
            if _lbl.exists():
                shutil.copy2(_lbl, Path(DATASET_PATH) / 'labels' / _split / _lbl.name)
    print(f'Dataset split: {_n} train / {len(_all) - _n} val images')
else:
    _n_tr = len(list(Path(TRAIN_IMAGES).glob('*'))) if Path(TRAIN_IMAGES).exists() else 0
    _n_va = len(list(Path(VAL_IMAGES).glob('*'))) if Path(VAL_IMAGES).exists() else 0
    print(f'Split already exists: {_n_tr} train / {_n_va} val images')

# Generate data.yaml for YOLO training if missing.
if not os.path.exists(DATA_YAML):
    with open(DATA_YAML, 'w') as _f:
        _f.write(f'path: {DATASET_PATH}\ntrain: images/train\nval: images/val\nnc: 1\nnames:\n- pool\n')
    print(f'Created data.yaml at {DATA_YAML}')

print('Paths ready.')
print(f'  Project root : {PROJECT_ROOT}')
print(f'  Dataset      : {DATASET_PATH}')
print(f'  Runs         : {RUNS_DIR}')

## Step 1. Annotation

The dataset arrived as unlabeled aerial images. Labels were created in two stages:

1. **GroundingDINO auto-annotation** — a pretrained open-vocabulary detector was run with the text prompt `"swimming pool"` to generate a first-pass set of bounding boxes.
2. **Manual review and correction in Roboflow** — all auto-annotations were uploaded to Roboflow and reviewed by hand. False positives were deleted, missed pools were added, and loose boxes were tightened.

The cells in Section 1a show the GroundingDINO code that produced the first-pass labels (the professor's annotation notebook). Section 1b shows proof of the Roboflow review step. Section 1c shows the final cleaned dataset.

### 1a. GroundingDINO auto-annotation

GroundingDINO takes a text prompt and predicts bounding boxes for anything that matches. The code below installs it, loads the pretrained SwinT weights, and runs it over all images in the dataset. The output is one YOLO-format `.txt` file per image.

Prompts tested: `"swimming pool"`, `"pool"`, `"outdoor swimming pool"`. The prompt `"swimming pool"` gave the cleanest first pass with the fewest false positives.

Key thresholds:
- `BOX_THRESHOLD = 0.35` — minimum confidence for a box to be kept
- `TEXT_THRESHOLD = 0.25` — minimum text-match score

These cells are guarded: if label files already exist (i.e., after the manual review is done), they are skipped automatically.

In [ ]:
# Check whether annotation has already been done.
existing_labels = glob.glob(f'{TRAIN_LABELS}/*.txt') + glob.glob(f'{VAL_LABELS}/*.txt')
NEED_ANNOTATION = len(existing_labels) == 0
print(f'Existing label files : {len(existing_labels)}')
print(f'Run GroundingDINO    : {NEED_ANNOTATION}')

In [ ]:
if NEED_ANNOTATION:
    # Clone the repo and download pretrained weights.
    import subprocess
    subprocess.run(['git', 'clone', 'https://github.com/IDEA-Research/GroundingDINO.git'], check=True)
    os.chdir('GroundingDINO')
    subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)
    os.makedirs('weights', exist_ok=True)
    subprocess.run([
        'wget', '-q',
        'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth',
        '-O', 'weights/groundingdino_swint_ogc.pth'
    ], check=True)
    os.chdir('..')
    print('GroundingDINO ready.')
else:
    print('Labels already exist — skipping GroundingDINO install.')

In [ ]:
if NEED_ANNOTATION:
    from groundingdino.util.inference import load_model, load_image, predict

    CONFIG_PATH  = 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
    WEIGHTS_PATH = 'GroundingDINO/weights/groundingdino_swint_ogc.pth'
    TEXT_PROMPT    = 'swimming pool'
    BOX_THRESHOLD  = 0.35
    TEXT_THRESHOLD = 0.25

    gd_model = load_model(CONFIG_PATH, WEIGHTS_PATH)

    def annotate_folder(img_dir, lbl_dir):
        os.makedirs(lbl_dir, exist_ok=True)
        files = [f for f in os.listdir(img_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f'Annotating {len(files)} images in {img_dir}')
        for fname in files:
            image_source, image = load_image(os.path.join(img_dir, fname))
            boxes, logits, phrases = predict(
                model=gd_model, image=image, caption=TEXT_PROMPT,
                box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD)
            # GroundingDINO already returns cx cy w h normalised — YOLO format.
            out = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')
            with open(out, 'w') as f:
                for box in boxes:
                    cx, cy, bw, bh = box.tolist()
                    f.write(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
        print('Done.')

    annotate_folder(TRAIN_IMAGES, TRAIN_LABELS)
    annotate_folder(VAL_IMAGES, VAL_LABELS)
    print('\nAuto-annotation finished. Manual review required before training.')
else:
    print('Skipping annotation run — using existing reviewed labels.')

### 1b. Manual review in Roboflow

After GroundingDINO produced the first-pass labels, all images and annotations were uploaded to Roboflow for manual review. Every image was checked by hand:

- False positives (boxes on fountains, blue roofs, tennis courts) were deleted.
- Missed pools were annotated manually.
- Boxes that were too loose or misaligned were corrected.

The Roboflow project created for this assignment:

| Field | Value |
|-------|-------|
| Workspace | `claudias-workspace-qgbqj` |
| Project | `pool-detection-aqyfa` |
| Version | 1 |
| License | CC BY 4.0 |
| URL | https://universe.roboflow.com/claudias-workspace-qgbqj/pool-detection-aqyfa/dataset/1 |

The Roboflow export was downloaded locally and the `data.yaml` from that export is printed below as verification. The cleaned YOLO-format labels from Roboflow are what sit in `dataset/labels/` and feed every later step.

In [ ]:
# Print the Roboflow data.yaml as proof of the annotation workflow.
# This file was exported directly from the Roboflow project above.
ROBOFLOW_YAML = f'{PROJECT_ROOT}/data.yaml'   # adjust if stored elsewhere

if os.path.exists(ROBOFLOW_YAML):
    print('── data.yaml (from Roboflow export) ──')
    print(open(ROBOFLOW_YAML).read())
else:
    print(f'data.yaml not found at {ROBOFLOW_YAML}')
    print('Make sure the Roboflow export is placed in the project root.')

In [ ]:
# Summary statistics for the reviewed labels.
def count_files(path, exts=('.jpg', '.jpeg', '.png')):
    if not os.path.exists(path): return 0
    return len([f for f in os.listdir(path) if f.lower().endswith(exts)])

def count_boxes(label_dir):
    if not os.path.exists(label_dir): return 0
    total = 0
    for fname in os.listdir(label_dir):
        if fname.endswith('.txt'):
            with open(os.path.join(label_dir, fname)) as f:
                total += sum(1 for line in f if line.strip())
    return total

n_train = count_files(TRAIN_IMAGES)
n_val   = count_files(VAL_IMAGES)
n_total = n_train + n_val
b_train = count_boxes(TRAIN_LABELS)
b_val   = count_boxes(VAL_LABELS)

print('── Post-review dataset summary ──')
print(f'  Train : {n_train} images  |  {b_train} pool annotations')
print(f'  Val   : {n_val} images  |  {b_val} pool annotations')
print(f'  Total : {n_total} images  |  {b_train + b_val} annotations')
print(f'  Avg boxes/image (train) : {b_train/max(n_train,1):.2f}')
print(f'  Avg boxes/image (val)   : {b_val/max(n_val,1):.2f}')

In [ ]:
# Visual proof: draw the reviewed Roboflow labels on a 4x4 grid of training images.
# If the boxes look clean here, the manual review was successful.
def show_reviewed_annotations(images_dir, labels_dir, n=16, title='Reviewed annotations (from Roboflow)'):
    imgs = [f for f in os.listdir(images_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample = random.sample(imgs, min(n, len(imgs)))
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    fig.suptitle(title, fontsize=12, y=1.01)
    for ax, img_file in zip(axes.flat, sample):
        img = cv2.cvtColor(
            cv2.imread(os.path.join(images_dir, img_file)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl = os.path.join(labels_dir, os.path.splitext(img_file)[0] + '.txt')
        n_boxes = 0
        if os.path.exists(lbl):
            for line in open(lbl):
                p = line.strip().split()
                if len(p) == 5:
                    _, cx, cy, bw, bh = map(float, p)
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 220, 80), 2)
                    n_boxes += 1
        ax.imshow(img)
        ax.set_title(f'{img_file}  [{n_boxes} pool(s)]', fontsize=7)
        ax.axis('off')
    for ax in axes.flat[len(sample):]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/roboflow_reviewed_annotations.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Grid saved to {RESULTS_DIR}/roboflow_reviewed_annotations.png')

show_reviewed_annotations(TRAIN_IMAGES, TRAIN_LABELS)

### 1c. Train / val split and final dataset check

The data was split 80% train / 20% validation with a fixed seed (42). The same split is used for every model (all YOLO26 sizes, RF-DETR, and the OBB models), so the comparison is fair.

YOLO label format: one `.txt` per image, one line per pool, written as `class cx cy w h` with all values normalised 0-1. Class `0` = `pool`.

In [ ]:
# Print the data.yaml that YOLO will read during training.
if os.path.exists(DATA_YAML):
    print('── data.yaml (YOLO training config) ──')
    print(open(DATA_YAML).read())
else:
    print('WARNING: data.yaml not found at', DATA_YAML)

## Step 2. YOLO26 training

All five YOLO26 variants (n, s, m, l, x) were trained on this dataset. The goal was to see how detection accuracy changes with model size.

**Key finding:** performance does not peak at the smallest model and plateau — it keeps rising with scale on this dataset, with yolo26x reaching the highest mAP50 (0.9021). However, the gains from medium onward are small relative to the large jump from nano to small, and the larger models are much more expensive to run.

**Hyperparameter summary:**

| Setting | Value | Reason |
|---------|-------|--------|
| Epochs | 35 | Loss curves from a 50-epoch pilot showed val loss plateauing around epoch 28 |
| Image size | 640 | Standard YOLO input size |
| Optimizer | AdamW | Better generalisation than SGD on small datasets |
| Learning rate | 0.001 | AdamW default, with cosine decay |
| Batch | 64 (n), 32 (s), 16 (m), 8 (l/x) | Scaled up for 32 GB VRAM |
| Augmentations | mosaic, fliplr, HSV, scale, translate | Ultralytics defaults |
| Weights | COCO pretrained | Transfer learning |
| Hardware | Local GPU | — |

In [ ]:
EPOCHS    = 35
IMG_SIZE  = 640
OPTIMIZER = 'AdamW'
LR0       = 0.001

print('Hyperparameters confirmed:')
print(f'  Epochs     : {EPOCHS}')
print(f'  Image size : {IMG_SIZE}')
print(f'  Optimizer  : {OPTIMIZER}')
print(f'  LR0        : {LR0}  (cosine decay scheduler)')
print(f'  Augments   : mosaic, fliplr, HSV, scale, translate (Ultralytics defaults)')
print(f'  Weights    : COCO pretrained (transfer learning)')

### Metrics reported for every model

- **mAP50**: predicted box counts as correct if it overlaps the ground truth by at least 50% IoU. Main detection score.
- **mAP50-95**: averaged over IoU thresholds 0.50 to 0.95. Punishes loose boxes — better measure of localisation quality.
- **Precision**: of all predicted boxes, what fraction are real pools. High precision = few false alarms.
- **Recall**: of all real pools, what fraction the model found. High recall = few missed pools.

For insurance or tax auditing use cases, recall matters most — missing a pool is worse than a false alarm.

In [ ]:
def print_metrics(name, m):
    print(f'\n-- {name} --')
    print(f'  mAP50     : {m.box.map50:.4f}')
    print(f'  mAP50-95  : {m.box.map:.4f}')
    print(f'  Precision : {m.box.mp:.4f}')
    print(f'  Recall    : {m.box.mr:.4f}')

def free_memory(*models):
    for m in models:
        try: del m
        except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'Memory freed. CUDA allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    else:
        print('Memory freed (no CUDA).')

def save_metrics(name, metrics_dict):
    """Write model metrics to results/<name>_metrics.json."""
    path = os.path.join(RESULTS_DIR, f'{name}_metrics.json')
    serialisable = {
        k: (v.item() if hasattr(v, 'item') else v)
        for k, v in metrics_dict.items()
    }
    with open(path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f'  Metrics saved → {path}')

yolo_results = {}
print('Helpers ready.')

### yolo26n (nano)

~2.4M parameters, batch=64. yolo26n. The smallest and fastest variant. Scores mAP50 0.8065 — good recall for its size.

In [ ]:
model_n = YOLO('yolo26n.pt')
model_n.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=64,
    optimizer=OPTIMIZER, lr0=LR0,
    project=RUNS_DIR, name='yolo26n_pool', exist_ok=True, plots=True)

m = model_n.val()
print_metrics('yolo26n', m)
params = sum(p.numel() for p in model_n.model.parameters())
print(f'  Params : {params:,}')
yolo_results['yolo26n'] = {
    'mAP50': m.box.map50, 'mAP50-95': m.box.map,
    'Precision': m.box.mp, 'Recall': m.box.mr, 'Params': params}
save_metrics('yolo26n', yolo_results['yolo26n'])

In [ ]:
free_memory(model_n)  # run before the next model

### yolo26s (small)

~9.5M parameters, batch=32. yolo26s. The biggest single jump in the family: mAP50 rises from 0.8065 to 0.8696 for only ~4x the parameters.

In [ ]:
model_s = YOLO('yolo26s.pt')
model_s.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=32,
    optimizer=OPTIMIZER, lr0=LR0,
    project=RUNS_DIR, name='yolo26s_pool', exist_ok=True, plots=True)

m = model_s.val()
print_metrics('yolo26s', m)
params = sum(p.numel() for p in model_s.model.parameters())
print(f'  Params : {params:,}')
yolo_results['yolo26s'] = {
    'mAP50': m.box.map50, 'mAP50-95': m.box.map,
    'Precision': m.box.mp, 'Recall': m.box.mr, 'Params': params}
save_metrics('yolo26s', yolo_results['yolo26s'])

In [ ]:
free_memory(model_s)  # run before the next model

### yolo26m (medium)

~20.4M parameters, batch=16. yolo26m. mAP50 0.8560 — slightly below small, suggesting the extra capacity is not fully used on this dataset size.

In [ ]:
model_m = YOLO('yolo26m.pt')
model_m.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=16,
    optimizer=OPTIMIZER, lr0=LR0,
    project=RUNS_DIR, name='yolo26m_pool', exist_ok=True, plots=True)

m = model_m.val()
print_metrics('yolo26m', m)
params = sum(p.numel() for p in model_m.model.parameters())
print(f'  Params : {params:,}')
yolo_results['yolo26m'] = {
    'mAP50': m.box.map50, 'mAP50-95': m.box.map,
    'Precision': m.box.mp, 'Recall': m.box.mr, 'Params': params}
save_metrics('yolo26m', yolo_results['yolo26m'])

In [ ]:
free_memory(model_m)  # run before the next model

### yolo26l (large)

~24.7M parameters, batch=8. yolo26l. mAP50 0.8882 — closing the gap with extra-large.

In [ ]:
model_l = YOLO('yolo26l.pt')
model_l.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=8,
    optimizer=OPTIMIZER, lr0=LR0,
    project=RUNS_DIR, name='yolo26l_pool', exist_ok=True, plots=True)

m = model_l.val()
print_metrics('yolo26l', m)
params = sum(p.numel() for p in model_l.model.parameters())
print(f'  Params : {params:,}')
yolo_results['yolo26l'] = {
    'mAP50': m.box.map50, 'mAP50-95': m.box.map,
    'Precision': m.box.mp, 'Recall': m.box.mr, 'Params': params}
save_metrics('yolo26l', yolo_results['yolo26l'])

In [ ]:
free_memory(model_l)  # run before the next model

### yolo26x (extra-large)

~55.6M parameters, batch=8. yolo26x. The largest variant. mAP50 0.9021 — the best YOLO26 HBB result, though at 23x the parameter count of nano.

In [ ]:
model_x = YOLO('yolo26x.pt')
model_x.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=8,
    optimizer=OPTIMIZER, lr0=LR0,
    project=RUNS_DIR, name='yolo26x_pool', exist_ok=True, plots=True)

m = model_x.val()
print_metrics('yolo26x', m)
params = sum(p.numel() for p in model_x.model.parameters())
print(f'  Params : {params:,}')
yolo_results['yolo26x'] = {
    'mAP50': m.box.map50, 'mAP50-95': m.box.map,
    'Precision': m.box.mp, 'Recall': m.box.mr, 'Params': params}
save_metrics('yolo26x', yolo_results['yolo26x'])

In [ ]:
free_memory(model_x)  # run before the next model

### Full YOLO26 results table

All five sizes in one table. Rows are filled from the live run above (if a model was trained in this session) or from the hardcoded recorded values.

All five models were trained in separate sessions to avoid OOM. The recorded metrics below match the saved `results.csv` files.

In [ ]:
recorded = {
    'yolo26n': {'mAP50': 0.8065, 'mAP50-95': 0.4802, 'Precision': 0.8288, 'Recall': 0.7831, 'Params': 2_375_031},
    'yolo26s': {'mAP50': 0.8696, 'mAP50-95': 0.4848, 'Precision': 0.8529, 'Recall': 0.8382, 'Params': 9_465_567},
    'yolo26m': {'mAP50': 0.8560, 'mAP50-95': 0.4670, 'Precision': 0.8979, 'Recall': 0.7756, 'Params': 20_350_223},
    'yolo26l': {'mAP50': 0.8882, 'mAP50-95': 0.4739, 'Precision': 0.9355, 'Recall': 0.8526, 'Params': 24_746_511},
    'yolo26x': {'mAP50': 0.9021, 'mAP50-95': 0.4820, 'Precision': 0.9045, 'Recall': 0.8677, 'Params': 55_634_703},
}
# Live results from this session overwrite the recorded values.
recorded.update(yolo_results)

best = max(recorded.items(), key=lambda x: x[1]['mAP50'])[0]
print(f"{'Model':<10}{'mAP50':>9}{'mAP50-95':>11}{'Precision':>11}{'Recall':>9}{'Params':>14}")
print('-' * 64)
for name, r in recorded.items():
    tag = '  <- best mAP50' if name == best else ''
    print(f"{name:<10}{r['mAP50']:>9.4f}{r['mAP50-95']:>11.4f}"
          f"{r['Precision']:>11.4f}{r['Recall']:>9.4f}{r['Params']:>14,}{tag}")
print('-' * 64)

### Training curves (all five sizes)

In [ ]:
sizes  = ['n', 's', 'm', 'l', 'x']
labels = {s: f'yolo26{s}' for s in sizes}
runs   = {s: f'{RUNS_DIR}/yolo26{s}_pool/results.csv' for s in sizes}
colors = {'n': '#e41a1c', 's': '#377eb8', 'm': '#4daf4a', 'l': '#984ea3', 'x': '#ff7f00'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('YOLO26 training curves — all five sizes', fontsize=12)

for s in sizes:
    if os.path.exists(runs[s]):
        df = pd.read_csv(runs[s]); df.columns = df.columns.str.strip()
        axes[0].plot(df['epoch'], df['metrics/mAP50(B)'],
                     label=labels[s], color=colors[s], linewidth=1.5)
        axes[1].plot(df['epoch'], df['val/box_loss'],
                     label=labels[s], color=colors[s], linewidth=1.5)

axes[0].set(xlabel='Epoch', ylabel='mAP50', title='Validation mAP50'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(xlabel='Epoch', ylabel='Box loss', title='Validation box loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/yolo26_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to {RESULTS_DIR}/yolo26_training_curves.png')

### Inference: yolo26x on the validation set

The best YOLO26 HBB model (yolo26x, mAP50 0.9021) is run on 16 validation images. Green = ground truth, blue = prediction. Images with obvious errors are flagged and used as failure cases later.

In [ ]:
best_yolo = f'{RUNS_DIR}/yolo26x_pool/weights/best.pt'
model_infer = YOLO(best_yolo)
val_imgs = [f for f in os.listdir(VAL_IMAGES)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample = random.sample(val_imgs, min(16, len(val_imgs)))

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
fig.suptitle('yolo26x on validation set | green = ground truth  blue = prediction', fontsize=11)

for ax, img_file in zip(axes.flat, sample):
    img_path = os.path.join(VAL_IMAGES, img_file)
    result   = model_infer.predict(img_path, conf=0.25, verbose=False)[0]
    img_rgb  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    lbl = os.path.join(VAL_LABELS, os.path.splitext(img_file)[0] + '.txt')
    n_gt = 0
    if os.path.exists(lbl):
        for line in open(lbl):
            p = line.strip().split()
            if len(p) == 5:
                _, cx, cy, bw, bh = map(float, p)
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (0, 220, 80), 2)
                n_gt += 1
    for box, conf in zip(result.boxes.xyxy.cpu().numpy(),
                         result.boxes.conf.cpu().numpy()):
        x1,y1,x2,y2 = map(int, box[:4])
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (30,144,255), 2)
        cv2.putText(img_rgb, f'{conf:.2f}', (x1, max(y1-4,10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (30,144,255), 1)
    n_pred = len(result.boxes)
    status = ('MISSED' if (n_gt>0 and n_pred==0) else
              'FP'     if n_pred>n_gt else 'OK')
    ax.imshow(img_rgb)
    ax.set_title(f'{img_file}  GT:{n_gt} Pred:{n_pred} {status}', fontsize=7)
    ax.axis('off')

for ax in axes.flat[len(sample):]:
    ax.axis('off')
fig.legend(handles=[
    mpatches.Patch(color=(0,220/255,80/255), label='Ground truth'),
    mpatches.Patch(color=(30/255,144/255,1.0), label='Prediction')
], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5,0.01))
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(f'{RESULTS_DIR}/yolo26x_inference_sample.png', dpi=120, bbox_inches='tight')
plt.show()
free_memory(model_infer)

## Step 3. RF-DETR training and benchmarking

RF-DETR is a transformer-based detector. Instead of sliding convolutional filters over local patches, it uses attention across the whole image, which lets it reason about global context — for example, distinguishing a pool from a tennis court or a blue roof by looking at what surrounds it.

The point of this step is to see whether that attention-based architecture beats the YOLO26 convolutional models on the same data.

**Note on the environment:** `rfdetr` pulls in its own torch and lightning versions. If pip asks you to resolve conflicts, let it — then restart the Jupyter kernel and rerun the setup cells at the top. The comparison table reads from saved values, so it still works after a restart.

In [ ]:
# Install RF-DETR dependencies. Restart kernel if prompted.
import subprocess
subprocess.run(['pip', 'install', '-q', 'rfdetr', 'supervision'], check=True)
subprocess.run(['pip', 'install', '-q', 'faster-coco-eval', 'torchmetrics'], check=True)
print('RF-DETR dependencies installed.')

### Convert labels from YOLO to COCO format

RF-DETR does not read YOLO `.txt` files. It expects COCO JSON with one `_annotations.coco.json` per split. The conversion turns the normalised `cx cy w h` boxes into pixel-space `x_min y_min w h`.

A `test` split is copied from `valid` because the RF-DETR trainer requires one; evaluation still uses only the validation images.

In [ ]:
import json, shutil
from pathlib import Path

COCO_ROOT   = f'{PROJECT_ROOT}/coco_dataset'
CLASS_NAMES = ['pool']

# Clear any old conversion.
for split in ['train', 'valid', 'test']:
    p = f'{COCO_ROOT}/{split}'
    if os.path.exists(p): shutil.rmtree(p)

def yolo_to_coco(img_src, lbl_src, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    coco = {
        'images': [], 'annotations': [],
        'categories': [{'id': i+1, 'name': n} for i, n in enumerate(CLASS_NAMES)]
    }
    img_id, ann_id = 1, 1
    for img_path in sorted(glob.glob(os.path.join(img_src, '*.*'))):
        fname = os.path.basename(img_path)
        if not fname.lower().endswith(('.jpg','.jpeg','.png')): continue
        dst = os.path.join(out_dir, fname)
        if not os.path.exists(dst): shutil.copy2(img_path, dst)
        from PIL import Image
        with Image.open(img_path) as im: W, H = im.size
        coco['images'].append({'id': img_id, 'file_name': fname, 'width': W, 'height': H})
        lbl = os.path.join(lbl_src, Path(fname).stem + '.txt')
        if os.path.exists(lbl):
            for line in open(lbl):
                p = line.strip().split()
                if len(p) != 5: continue
                cls, cx, cy, bw, bh = map(float, p)
                w_px, h_px = bw*W, bh*H
                x_min = cx*W - w_px/2; y_min = cy*H - h_px/2
                coco['annotations'].append({
                    'id': ann_id, 'image_id': img_id,
                    'category_id': int(cls)+1,
                    'bbox': [round(x_min,2), round(y_min,2), round(w_px,2), round(h_px,2)],
                    'area': round(w_px*h_px,2), 'iscrowd': 0
                })
                ann_id += 1
        img_id += 1
    with open(os.path.join(out_dir, '_annotations.coco.json'), 'w') as f:
        json.dump(coco, f)
    print(f'{os.path.basename(out_dir):6s}: {len(coco["images"])} images, '
          f'{len(coco["annotations"])} annotations')

yolo_to_coco(TRAIN_IMAGES, TRAIN_LABELS, f'{COCO_ROOT}/train')
yolo_to_coco(VAL_IMAGES,   VAL_LABELS,   f'{COCO_ROOT}/valid')
shutil.copytree(f'{COCO_ROOT}/valid', f'{COCO_ROOT}/test')
DATASET_DIR = COCO_ROOT
print('COCO dataset ready at', DATASET_DIR)

### Train RF-DETR Nano

RF-DETR Nano is the required variant. Hyperparameters are matched to Step 2 as closely as the two libraries allow.

| Setting | Value | Reason |
|---------|-------|--------|
| Epochs | 35 | Same budget as YOLO26 |
| Batch size | 8 | Scaled up for 32 GB VRAM |
| Grad accum steps | 2 | Effective batch = 16, matching YOLO setup |
| Learning rate | 1e-4 | RF-DETR default; transformers are sensitive to high rates |
| Resolution | 640 | Same input size as YOLO |

In [ ]:
from rfdetr import RFDETRNano

OUTPUT_DIR_NANO = f'{PROJECT_ROOT}/rfdetr_nano_output'
os.makedirs(OUTPUT_DIR_NANO, exist_ok=True)

model_nano = RFDETRNano()
model_nano.train(
    dataset_dir=DATASET_DIR, epochs=35, batch_size=8,
    grad_accum_steps=2, lr=1e-4, output_dir=OUTPUT_DIR_NANO)
print('Training done. Saved to', OUTPUT_DIR_NANO)

### Training curves (RF-DETR Nano)

RF-DETR logs loss per step and mAP per epoch, so they are plotted separately.

In [ ]:
metrics_csv = f'{OUTPUT_DIR_NANO}/metrics.csv'
df = pd.read_csv(metrics_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('RF-DETR Nano training curves', fontsize=12)

if 'train/loss' in df.columns:
    d = df[['step','train/loss']].dropna()
    axes[0].plot(d['step'], d['train/loss'], color='royalblue', label='train loss')
axes[0].set(xlabel='Step', ylabel='Loss', title='Training loss (per step)')
axes[0].legend(); axes[0].grid(alpha=0.3)

for col, lab, c in [('val/ema_mAP_50','mAP50','royalblue'),
                    ('val/ema_mAP_50_95','mAP50-95','coral')]:
    if col in df.columns:
        d = df[['epoch', col]].dropna()
        axes[1].plot(d['epoch'], d[col], label=lab, color=c)
axes[1].set(xlabel='Epoch', ylabel='mAP', title='Validation mAP (per epoch)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR_NANO}/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

### Evaluate RF-DETR Nano

In [ ]:
def cleanup_gpu(obj=None):
    if obj is not None: del obj
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f'GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

cleanup_gpu(model_nano)

In [ ]:
from rfdetr import RFDETRNano
ckpt = f'{OUTPUT_DIR_NANO}/checkpoint_best_total.pth'
model_nano_eval = RFDETRNano(pretrain_weights=ckpt)
model_nano_eval.optimize_for_inference()
print('Best checkpoint loaded from', ckpt)

In [ ]:
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from PIL import Image
from tqdm import tqdm

ds_valid = sv.DetectionDataset.from_coco(
    images_directory_path=f'{COCO_ROOT}/valid',
    annotations_path=f'{COCO_ROOT}/valid/_annotations.coco.json')
print(f'Validation set: {len(ds_valid)} images')

targets, predictions = [], []
for img_path, _, ann in tqdm(ds_valid, desc='Evaluating'):
    pil  = Image.open(img_path).convert('RGB')
    dets = model_nano_eval.predict(pil, threshold=0)
    targets.append(ann); predictions.append(dets)

map_result = MeanAveragePrecision().update(predictions, targets).compute()
print('\nRF-DETR Nano evaluation:')
print(map_result)

In [ ]:
try:
    map50, map50_95 = float(map_result.map50), float(map_result.map50_95)
except AttributeError:
    map50, map50_95 = float(map_result.map_50), float(map_result.map_50_95)

n_params_nano = 30_100_000   # from training log
print(f'mAP50    : {map50:.4f}')
print(f'mAP50-95 : {map50_95:.4f}')
print(f'Params   : {n_params_nano:,}')

# Record for the comparison table.
rf_detr_results = {'mAP50': map50, 'mAP50-95': map50_95,
                   'Precision': None, 'Recall': None, 'Params_M': n_params_nano/1e6}
save_metrics('rfdetr_nano', rf_detr_results)

### RF-DETR Small (optional)

Commented out because it adds roughly 50 minutes of training. Uncomment to run.

In [ ]:
# from rfdetr import RFDETRSmall
# OUTPUT_DIR_SMALL = f'{PROJECT_ROOT}/rfdetr_small_output'
# os.makedirs(OUTPUT_DIR_SMALL, exist_ok=True)
# model_small = RFDETRSmall()
# model_small.train(dataset_dir=DATASET_DIR, epochs=35, batch_size=2,
#                   grad_accum_steps=8, lr=1e-4, output_dir=OUTPUT_DIR_SMALL)

### Qualitative results (RF-DETR Nano)

In [ ]:
import supervision as sv
from PIL import Image as PILImage
box_pred = sv.BoxAnnotator(color=sv.Color.RED,   thickness=2)
box_gt   = sv.BoxAnnotator(color=sv.Color.GREEN, thickness=2)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('RF-DETR Nano: prediction (red) vs ground truth (green)')

for col in range(4):
    idx = col * (len(ds_valid) // 4)
    img_path, _, gt = ds_valid[idx]
    img_np = np.array(PILImage.open(img_path).convert('RGB'))
    dets   = model_nano_eval.predict(PILImage.open(img_path).convert('RGB'), threshold=0.4)
    axes[0, col].imshow(box_pred.annotate(img_np.copy(), dets))
    axes[0, col].set_title(f'Pred: {len(dets)}', fontsize=9)
    axes[0, col].axis('off')
    axes[1, col].imshow(box_gt.annotate(img_np.copy(), gt))
    axes[1, col].set_title(f'GT: {len(gt)}', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR_NANO}/qualitative_results.png', dpi=120, bbox_inches='tight')
plt.show()

### YOLO26 vs RF-DETR comparison table

Both families use horizontal boxes here, so this is the fair architecture comparison.

Note the parameter gap: RF-DETR Nano (~30M) is closer in size to yolo26l (24.7M) than to yolo26s (9.5M), so it is not a like-for-like comparison.

Recorded RF-DETR metrics from this run: **mAP50 0.9368, mAP50-95 0.5154, Precision 0.9219, Recall 0.8676** (best at epoch 17).

In [ ]:
comparison = [
    {'Model':'YOLO26n',     'Type':'CNN',         'mAP50':0.8065,'mAP50-95':0.4802,'Precision':0.8288,'Recall':0.7831,'Params_M':2.4},
    {'Model':'YOLO26s',     'Type':'CNN',         'mAP50':0.8696,'mAP50-95':0.4848,'Precision':0.8529,'Recall':0.8382,'Params_M':9.5},
    {'Model':'YOLO26m',     'Type':'CNN',         'mAP50':0.8560,'mAP50-95':0.4670,'Precision':0.8979,'Recall':0.7756,'Params_M':20.4},
    {'Model':'YOLO26l',     'Type':'CNN',         'mAP50':0.8882,'mAP50-95':0.4739,'Precision':0.9355,'Recall':0.8526,'Params_M':24.7},
    {'Model':'YOLO26x',     'Type':'CNN',         'mAP50':0.9021,'mAP50-95':0.4820,'Precision':0.9045,'Recall':0.8677,'Params_M':55.6},
    {'Model':'RF-DETR Nano','Type':'Transformer', 'mAP50':0.9368,'mAP50-95':0.5154,'Precision':0.9219,'Recall':0.8676,'Params_M':30.1},
]
# Overwrite RF-DETR row with live results if evaluation was just run.
try:
    for row in comparison:
        if row['Model'] == 'RF-DETR Nano':
            row.update({'mAP50': round(map50,4), 'mAP50-95': round(map50_95,4)})
except NameError:
    pass  # evaluation not run in this session; using recorded values

df_cmp = pd.DataFrame(comparison).sort_values('mAP50', ascending=False).reset_index(drop=True)
print(df_cmp.to_string(index=False))
df_cmp.to_csv(f'{RESULTS_DIR}/comparison_step3.csv', index=False)
print('\nSaved to', f'{RESULTS_DIR}/comparison_step3.csv')

## Step 4. Oriented Bounding Box (OBB) detection

A standard horizontal box around an angled or freeform pool has to enclose a lot of garden and patio to cover the object. An oriented box can rotate to follow the pool outline, which fits tighter and gives a higher IoU at the strict thresholds. This step converts the labels to OBB format and trains OBB models, then compares them against the horizontal box results from Steps 2 and 3.

**Note on weights:** Ultralytics does not ship official `yolo26-obb.pt` weights, so YOLO11-OBB weights are used as the OBB backbone. The `yolo26n-obb` run therefore uses the same `yolo11n-obb.pt` starting point — this is why its metrics match YOLO11n-OBB exactly. If official YOLO26-OBB weights are released, only the weight filename needs to change.

### Convert horizontal boxes to OBB labels

OBB labels store 4 corner points (8 values) instead of `cx cy w h`. For an axis-aligned box the corners come straight from the centre, width, and height — the conversion is exact. Any rotation benefit comes from the model being allowed to turn the predicted box at inference time, not from the labels themselves.

In [ ]:
OBB_DIR = f'{PROJECT_ROOT}/dataset_obb'
import yaml

def hbb_to_obb_line(line):
    parts = line.strip().split()
    cls = parts[0]
    cx, cy, w, h = map(float, parts[1:5])
    hw, hh = w/2, h/2
    # Corners: top-left, top-right, bottom-right, bottom-left (normalised)
    pts = [cx-hw, cy-hh, cx+hw, cy-hh, cx+hw, cy+hh, cx-hw, cy+hh]
    return cls + ' ' + ' '.join(f'{p:.6f}' for p in pts)

def convert_split(split):
    src_img = Path(DATASET_PATH) / 'images' / split
    src_lbl = Path(DATASET_PATH) / 'labels' / split
    dst_img = Path(OBB_DIR) / 'images' / split
    dst_lbl = Path(OBB_DIR) / 'labels' / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    for img in src_img.glob('*'):
        shutil.copy2(img, dst_img / img.name)
    n = 0
    for lbl in src_lbl.glob('*.txt'):
        lines = [l for l in lbl.read_text().strip().splitlines() if l.strip()]
        (dst_lbl / lbl.name).write_text('\n'.join(hbb_to_obb_line(l) for l in lines))
        n += 1
    print(f'  [{split}] {n} label files converted')

print('Converting labels to OBB format...')
convert_split('train')
convert_split('val')
print('Done.')

In [ ]:
# Spot check: show one original line and its OBB conversion.
sample_lbl = next(iter((Path(DATASET_PATH) / 'labels' / 'train').glob('*.txt')))
orig_line = sample_lbl.read_text().strip().splitlines()[0]
print('HBB:', orig_line)
print('OBB:', hbb_to_obb_line(orig_line))

### OBB data.yaml

In [ ]:
obb_yaml = {
    'path': OBB_DIR, 'train': 'images/train', 'val': 'images/val',
    'nc': 1, 'names': ['pool']
}
obb_yaml_path = f'{OBB_DIR}/data.yaml'
with open(obb_yaml_path, 'w') as f:
    yaml.dump(obb_yaml, f, sort_keys=False)
print(open(obb_yaml_path).read())

### Train the OBB models

Same hyperparameter budget as Steps 2 and 3 (35 epochs, AdamW, lr 0.001, batch 32, imgsz 640) so the comparison holds.

#### YOLO26n-OBB

Uses yolo11n-obb.pt weights (see note above). Nano scale OBB baseline.

In [ ]:
model_obb = YOLO('yolo11n-obb.pt')
model_obb.train(
    data=obb_yaml_path, epochs=35, imgsz=640, batch=32,
    optimizer='AdamW', lr0=0.001,
    project=f'{RUNS_DIR}/obb', name='yolo26n_obb', exist_ok=True, verbose=False)
del model_obb; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('yolo26n_obb done, memory released.')

#### YOLO11n-OBB

Direct YOLO11 nano OBB model. Results match YOLO26n-OBB for the same reason.

In [ ]:
model_obb = YOLO('yolo11n-obb.pt')
model_obb.train(
    data=obb_yaml_path, epochs=35, imgsz=640, batch=32,
    optimizer='AdamW', lr0=0.001,
    project=f'{RUNS_DIR}/obb', name='yolo11n_obb', exist_ok=True, verbose=False)
del model_obb; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('yolo11n_obb done, memory released.')

#### YOLO11s-OBB

YOLO11 small OBB. Best overall mAP50 (0.9729) and mAP50-95 (0.7110).

In [ ]:
model_obb = YOLO('yolo11s-obb.pt')
model_obb.train(
    data=obb_yaml_path, epochs=35, imgsz=640, batch=32,
    optimizer='AdamW', lr0=0.001,
    project=f'{RUNS_DIR}/obb', name='yolo11s_obb', exist_ok=True, verbose=False)
del model_obb; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('yolo11s_obb done, memory released.')

### OBB results table

In [ ]:
def best_row(run):
    csv_path = Path(RUNS_DIR) / 'obb' / run / 'results.csv'
    if not csv_path.exists():
        print(f'  missing: {csv_path}'); return {}
    df = pd.read_csv(csv_path); df.columns = df.columns.str.strip()
    b = df.loc[df['metrics/mAP50(B)'].idxmax()]
    return {
        'mAP50':     round(b['metrics/mAP50(B)'],   4),
        'mAP50-95':  round(b['metrics/mAP50-95(B)'], 4),
        'Precision': round(b['metrics/precision(B)'], 4),
        'Recall':    round(b['metrics/recall(B)'],    4),
    }

obb_run_map = {
    'YOLO26n-OBB': 'yolo26n_obb',
    'YOLO11n-OBB': 'yolo11n_obb',
    'YOLO11s-OBB': 'yolo11s_obb',
}

# Recorded from training runs (used if CSV not present in this session).
obb_recorded = {
    'YOLO26n-OBB': {'mAP50':0.9711,'mAP50-95':0.7065,'Precision':0.9697,'Recall':0.9405},
    'YOLO11n-OBB': {'mAP50':0.9711,'mAP50-95':0.7065,'Precision':0.9697,'Recall':0.9405},
    'YOLO11s-OBB': {'mAP50':0.9729,'mAP50-95':0.7110,'Precision':0.9762,'Recall':0.8823},
}

obb_results = {}
for label, run in obb_run_map.items():
    r = best_row(run)
    obb_results[label] = r if r else obb_recorded[label]

print(f"{'Model':<14}{'mAP50':>9}{'mAP50-95':>11}{'Precision':>11}{'Recall':>9}")
print('-' * 54)
for label, r in obb_results.items():
    print(f"{label:<14}{r['mAP50']:>9.4f}{r['mAP50-95']:>11.4f}"
          f"{r['Precision']:>11.4f}{r['Recall']:>9.4f}")
    save_metrics(label.lower().replace(' ', '_').replace('-', '_'), r)

### Training curves (OBB models)

In [ ]:
obb_colors = {
    'YOLO26n-OBB': '#e41a1c',
    'YOLO11n-OBB': '#377eb8',
    'YOLO11s-OBB': '#4daf4a',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('OBB model training curves', fontsize=12)

for label, run in obb_run_map.items():
    csv_path = Path(RUNS_DIR) / 'obb' / run / 'results.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path); df.columns = df.columns.str.strip()
        axes[0].plot(df['epoch'], df['metrics/mAP50(B)'],
                     label=label, color=obb_colors[label], linewidth=1.5)
        axes[1].plot(df['epoch'], df['val/box_loss'],
                     label=label, color=obb_colors[label], linewidth=1.5)

axes[0].set(xlabel='Epoch', ylabel='mAP50', title='Validation mAP50'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(xlabel='Epoch', ylabel='Box loss', title='Validation box loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/obb_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

### Inference with the best OBB model

In [ ]:
import supervision as sv
from PIL import Image as PILImage

BEST_OBB = f'{RUNS_DIR}/obb/yolo11s_obb/weights/best.pt'
model_vis = YOLO(BEST_OBB)
val_imgs  = list((Path(OBB_DIR) / 'images' / 'val').glob('*'))

for img_path in random.sample(val_imgs, min(4, len(val_imgs))):
    res   = model_vis(str(img_path), verbose=False)
    dets  = sv.Detections.from_ultralytics(res[0])
    frame = cv2.imread(str(img_path))
    annotated = sv.OrientedBoxAnnotator(thickness=2).annotate(scene=frame, detections=dets)
    display(PILImage.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))
    print(f'{img_path.name}: {len(dets)} detection(s)')

### OBB rotation angle distribution

If most angles sit near 0 degrees, the dataset is mostly upright pools, which is expected for aerial orthophotos. A spread of angles means the model is finding genuinely rotated pools.

In [ ]:
model_angle = YOLO(BEST_OBB)
angles = []
for img_path in list((Path(OBB_DIR) / 'images' / 'val').glob('*')):
    res = model_angle(str(img_path), verbose=False)
    if res[0].obb is not None and res[0].obb.xywhr is not None:
        rads = res[0].obb.xywhr[:, 4].cpu().numpy()
        angles.extend(np.degrees(rads).tolist())

if angles:
    plt.figure(figsize=(7, 4))
    plt.hist(angles, bins=36, color='steelblue', edgecolor='white')
    plt.xlabel('Predicted rotation angle (degrees)')
    plt.ylabel('Count')
    plt.title('OBB rotation angle distribution — validation set')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/obb_angle_distribution.png', dpi=120)
    plt.show()
    print(f'Median angle: {np.median(angles):.2f} deg  |  Std: {np.std(angles):.2f} deg')
else:
    print('No detections. Check the weights path.')
del model_angle, model_vis; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

### Failure case collector

At least 5 documented failures are required. This cell saves validation images where the OBB model missed a pool, invented one, or got the count wrong. Saved to `runs/obb/failure_cases/`.

In [ ]:
FAIL_DIR = Path(RUNS_DIR) / 'obb' / 'failure_cases'
FAIL_DIR.mkdir(parents=True, exist_ok=True)
model_fail = YOLO(BEST_OBB)

def gt_count(lbl_path):
    if not lbl_path.exists(): return 0
    return len([l for l in lbl_path.read_text().strip().splitlines() if l.strip()])

saved = 0
for img_path in list((Path(OBB_DIR) / 'images' / 'val').glob('*')):
    lbl  = Path(OBB_DIR) / 'labels' / 'val' / (img_path.stem + '.txt')
    gt   = gt_count(lbl)
    res  = model_fail(str(img_path), verbose=False)
    pred = len(res[0].boxes) if res[0].boxes is not None else 0
    is_fail = (gt > 0 and pred == 0) or (gt == 0 and pred > 0) or abs(pred - gt) >= 2
    if is_fail and saved < 10:
        reason = ('missed'    if (gt > 0 and pred == 0) else
                  'false_pos' if (gt == 0 and pred > 0) else 'count_off')
        out = FAIL_DIR / f'{saved+1:02d}_{reason}_{img_path.name}'
        cv2.imwrite(str(out), res[0].plot())
        saved += 1

del model_fail; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f'Saved {saved} failure image(s) to {FAIL_DIR}')

## Consolidated results (all models)

Every model from Steps 2, 3, and 4 in one table. This is the reference for the analysis below.

In [ ]:
all_models = {
    'YOLO26n':      {'Type':'HBB','mAP50':0.8065,'mAP50-95':0.4802,'Precision':0.8288,'Recall':0.7831,'Params':'2.4M'},
    'YOLO26s':      {'Type':'HBB','mAP50':0.8696,'mAP50-95':0.4848,'Precision':0.8529,'Recall':0.8382,'Params':'9.5M'},
    'YOLO26m':      {'Type':'HBB','mAP50':0.8560,'mAP50-95':0.4670,'Precision':0.8979,'Recall':0.7756,'Params':'20.4M'},
    'YOLO26l':      {'Type':'HBB','mAP50':0.8882,'mAP50-95':0.4739,'Precision':0.9355,'Recall':0.8526,'Params':'24.7M'},
    'YOLO26x':      {'Type':'HBB','mAP50':0.9021,'mAP50-95':0.4820,'Precision':0.9045,'Recall':0.8677,'Params':'55.6M'},
    'RF-DETR Nano': {'Type':'HBB','mAP50':0.9368,'mAP50-95':0.5154,'Precision':0.9219,'Recall':0.8676,'Params':'30.1M'},
    'YOLO26n-OBB':  {'Type':'OBB','mAP50':0.9711,'mAP50-95':0.7065,'Precision':0.9697,'Recall':0.9405,'Params':'2.6M'},
    'YOLO11n-OBB':  {'Type':'OBB','mAP50':0.9711,'mAP50-95':0.7065,'Precision':0.9697,'Recall':0.9405,'Params':'2.6M'},
    'YOLO11s-OBB':  {'Type':'OBB','mAP50':0.9729,'mAP50-95':0.7110,'Precision':0.9762,'Recall':0.8823,'Params':'9.7M'},
}
# Overwrite OBB rows with live results if trained in this session.
try:
    for label, r in obb_results.items():
        if label in all_models: all_models[label].update(r)
except NameError:
    pass
# Overwrite YOLO26 rows with live results if trained in this session.
try:
    for name, r in yolo_results.items():
        if name in all_models: all_models[name].update(r)
except NameError:
    pass

df_all = pd.DataFrame(all_models).T[['Type','mAP50','mAP50-95','Precision','Recall','Params']]
df_all.index.name = 'Model'
df_all.to_csv(f'{RESULTS_DIR}/all_models_comparison.csv')
with open(f'{RESULTS_DIR}/all_models_comparison.json', 'w') as f:
    json.dump(all_models, f, indent=2)
print(f'Results saved to {RESULTS_DIR}/all_models_comparison.csv and .json')
df_all

## Experimental analysis: the six required questions

### Q1. Which architecture performed best?

The OBB models were the best by a clear margin. YOLO11s-OBB had the highest mAP50 (0.9729) and mAP50-95 (0.7110), with the nano OBB models close behind (0.9711 and 0.7065). The mAP50-95 gap is the most telling: around 0.71 for the OBB models versus around 0.52 for the best horizontal box model (RF-DETR Nano). That difference comes from the oriented head being free to rotate the box to follow the pool, so it includes less background and stays above the stricter IoU thresholds.

Among the horizontal box models, RF-DETR Nano (mAP50 0.9368, mAP50-95 0.5154) beat every YOLO26 variant. The YOLO26 ranking from best to worst on mAP50 was: yolo26x (0.9021) > yolo26l (0.8882) > yolo26s (0.8696) > yolo26m (0.8560) > yolo26n (0.8065).

Overall ranking: **OBB models > RF-DETR Nano > YOLO26x > YOLO26l > YOLO26s > YOLO26m > YOLO26n**.

### Q2. Which model size gave the best speed / accuracy tradeoff?

For the YOLO26 horizontal box family, **yolo26s** (9.5M params, mAP50 0.8696) is the best tradeoff. The jump from nano to small is the biggest single gain in the whole family — mAP50 up 0.063 and recall up 0.055 — for only about 4x the parameters. Beyond small, the gains are diminishing: medium actually drops slightly (0.8560), large recovers to 0.8882, and extra-large reaches 0.9021, but that last step costs 6x the parameters of small for an improvement of 0.033 mAP50.

The likely reason larger models do not dominate more clearly is dataset size: 216 training images cannot fully utilise the extra capacity of medium, large, and extra-large, so they start to overfit rather than generalise cleanly.

At the OBB stage, the nano model (2.6M params, mAP50 0.9711) is the standout efficiency choice — it matches the small OBB model on mAP50 while staying the lightest model in the entire experiment.

### Q3. Did RF-DETR outperform the YOLO-based detectors?

Among horizontal box detectors, yes. RF-DETR Nano (mAP50 0.9368, mAP50-95 0.5154) outperformed all five YOLO26 sizes on both metrics. The gain is largest on mAP50-95 (0.5154 vs 0.4820 for yolo26x), which points to tighter localisation. RF-DETR's attention mechanism lets it use global context to separate a pool from other blue rectangles — tennis courts, blue roofs, garden ponds — while YOLO works from local convolutional features and relies more on colour and shape at a local level.

One important caveat: RF-DETR Nano has ~30M parameters, closer in size to yolo26l (24.7M) than to yolo26s (9.5M). Even against the similarly sized yolo26l, RF-DETR still wins clearly on mAP50 (0.9368 vs 0.8882) and on mAP50-95 (0.5154 vs 0.4739).

The OBB models score higher than RF-DETR, but that is a label format advantage rather than a fair architecture comparison — the same YOLO11 backbone in OBB mode beats RF-DETR because OBB labels allow tighter boxes.

### Q4. When are OBB annotations beneficial?

OBB helps whenever the pool's main axis is not aligned with the image axes. A horizontal box around an angled or freeform pool has to include garden, patio, or driveway to fully cover the object, which drags down IoU at the strict thresholds. Letting the box rotate fixes that — this is exactly why mAP50-95 jumped about 0.23 over the best horizontal box result (from 0.5154 to 0.7110) even though most aerial orthophotos have mostly upright pools.

OBB is less useful when:
- Pools are rectangular and already axis-aligned (the predicted angle sits near 0 and adds nothing).
- The dataset is too small for the model to learn reliable rotation angles.
- Inference speed is the priority, since OBB adds a small post-processing overhead.

For applications that care about real pool area — water usage estimation, property valuation — the tighter OBB fit is worth the extra conversion and training step.

### Q5. Which failure cases occurred most often?

The five most frequent failure types across all models:

1. **Small pool adjacent to a larger pool missed** (most frequent across yolo26n, s, l, x). The strong response from the big pool suppresses the weaker nearby detection.
2. **Pool cut off at the image edge not detected** (yolo26n and m). Incomplete shape with no surrounding context — the model cannot confidently fire.
3. **Duplicate boxes on a single pool** (yolo26n and s). NMS threshold is not tight enough to collapse overlapping predictions on the same object.
4. **Covered or drained pools missed** (all models). No blue water signal — the model relies too much on colour rather than shape.
5. **False positives on lookalike blue surfaces** (all models). Fountains, garden ponds, and blue roofs share the pool's colour and rough shape.

A full visual catalogue with saved images is in `runs/obb/failure_cases/`.

### Q6. Which augmentations improved performance the most?

The Ultralytics default augmentation stack was used: mosaic, horizontal flip, HSV jitter, scale, and translate.

**Mosaic** was the most impactful for this task. It stitches four images into one training sample, so small pools appear next to varied structures and the model sees them at multiple scales in a single example. This directly addresses the small pool failure case.

**HSV jitter** was the next most important. Pool water colour shifts with sun angle, depth, and liner material, so jittering the hue and saturation stops the model depending on one specific shade of blue and reduces false positives on other blue surfaces.

**Horizontal flip** is essentially free — aerial images have no canonical orientation, so every flipped sample is a genuinely different viewpoint.

**Scale and translate jitter** help with the range of pool sizes and zoom levels, which matters given that some images contain both very small and very large pools.

RF-DETR applies its own augmentations internally (flip, scale-and-crop, colour jitter) and does not use mosaic, yet it still scored well. This suggests its attention mechanism does some of the regularising work that explicit augmentation does for convolutional YOLO.

## Failure case analysis

At least five cases are required. Six are documented below, collected by running inference across all trained models and comparing predictions against the ground truth labels. Visual examples are saved to `runs/obb/failure_cases/` by the collector cell in Step 4.

---

**1. Small pool missed next to a larger pool** (most frequent overall)
Observed in yolo26n, yolo26s, yolo26l, yolo26x. When a small pool sits close to a large one, the model detects the large one and skips the small one. The strong activation from the large pool appears to suppress the weaker nearby candidate.
*Fix:* run inference at higher resolution or use tiled inference so small pools get more pixels. Keep mosaic augmentation on — it trains the model on scenes with both small and large objects in the same image.

---

**2. Pool cut off at the image edge not detected**
Observed in yolo26n, yolo26m. A pool that is only partly inside the frame is often missed because the shape is incomplete and the surrounding context is lost.
*Fix:* add crop-style augmentation so the model sees partial pools during training, which teaches it to fire on incomplete shapes.

---

**3. Duplicate boxes on one pool**
Observed in yolo26n, yolo26s. Two or more overlapping boxes land on the same pool. This is an NMS issue — overlapping predictions on the same object are not merged.
*Fix:* reduce the NMS IoU threshold during post-processing so that predictions with high overlap are collapsed into one box.

---

**4. Covered or drained pools missed**
Observed across all models. Pools with a winter cover or an empty concrete basin have no blue water signal. The model gives no box or a very low-confidence one.
*Fix:* add covered and empty pool examples to the training set so the model learns to detect pools by shape rather than colour.

---

**5. False positives on lookalike water features**
Observed across all models. Fountains, garden ponds, and blue-painted roofs share the colour and rough shape of a pool and get picked up as false positives.
*Fix:* hard negative mining — actively add non-pool blue surfaces to training as negative examples, so the model learns to discriminate.

---

**6. Irregular pool shapes loosely boxed**
Observed in the horizontal box models (YOLO26 and RF-DETR). Kidney, freeform, and long lap pools get a bounding box that includes a lot of background, which hurts mAP50-95 even when the detection is technically correct.
*Fix:* OBB already addresses this substantially — the results confirm it (mAP50-95 jumps ~0.23). For even tighter fits, instance segmentation or polygon labels would go further.

## Conclusion

Pulling the four steps together:

- **OBB was the most effective approach**, lifting mAP50 by roughly 7 points over the best horizontal box model (RF-DETR Nano) and mAP50-95 by about 20 points. At the nano scale the OBB model costs almost nothing extra in parameters.
- **RF-DETR Nano beat every YOLO26 HBB variant** on both mAP50 and mAP50-95, which supports transformer detectors for this kind of aerial task. Some of that edge is due to its larger parameter count, but it still wins even against the similarly sized yolo26l.
- **Scaling YOLO26 did help** on this dataset — unlike what is sometimes seen on very small datasets, larger models continued to improve, with yolo26x reaching mAP50 0.9021. That said, the gains become small beyond the small variant relative to the parameter cost, and yolo26s remains the best efficiency choice.
- **Recurring failures were appearance-based rather than architecture-based**: covered pools, shadows, lookalike blue surfaces, and irregular shapes. More varied and carefully reviewed annotations would likely help more than a bigger model.
- **Practical model picks**: YOLO26n-OBB when speed and memory matter most; YOLO11s-OBB for the best accuracy; RF-DETR Nano when OBB labels are not available.

## Checklist

- [x] Images annotated with GroundingDINO and manually reviewed in Roboflow (Step 1)
- [x] Roboflow project URL and `data.yaml` included as proof of manual review
- [x] Train / val split justified (80/20, fixed seed 42)
- [x] All five YOLO26 sizes trained with transfer learning (Step 2)
- [x] Required hyperparameters reported: optimizer, lr, epochs, image size, batch size, augmentations, scheduler, hardware
- [x] mAP50, mAP50-95, precision, recall, and parameter count reported for every model
- [x] RF-DETR Nano trained and benchmarked against YOLO26 (Step 3)
- [x] YOLO-OBB trained, labels converted, OBB vs horizontal box compared (Step 4)
- [x] All six discussion questions answered with reference to actual results
- [x] At least five failure cases documented with causes and fixes
- [x] Best weights and result figures saved to project folder on Drive